# DINOv3 raw CLS retrieval — evaluation only

No training. Load the same DINOv3 experiment checkpoint used for the projection result, restore both backbones, then bypass the learned projection for BOTH queries and detected gallery crops. Cosine similarity uses normalized raw CLS features (384 dimensions for ViT-S/16). The existing projection and checkpoint are not modified.

Run setup, set DINO_CHECKPOINT below, and run evaluation. Same crop preprocessing and detection threshold 0.5. Results go to a new W&B evaluation run with explicit raw-CLS provenance, final-score table/chart and retrieval previews. First use still requires Internet for the official Hub implementation; the checkpoint supplies all model weights. No original DINO pretrained weights file is required.


In [ ]:
from kaggle_secrets import UserSecretsClient
import base64
import os
import subprocess
import sys
from pathlib import Path

USER = "SaadQAbbasi"
REPO = "ML4CV-Assignment"
repo_path = f"/kaggle/working/{REPO}"
repo_dir = Path(repo_path)
remote = f"https://github.com/{USER}/{REPO}.git"

# Pass credentials through the child environment, not the URL or notebook output.
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
git_env = os.environ.copy()
git_env.update({
    "GIT_TERMINAL_PROMPT": "0",
    "GIT_CONFIG_COUNT": "1",
    "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
    "GIT_CONFIG_VALUE_0": f"Authorization: Basic {auth}",
})
try:
    if not repo_dir.exists():
        command = ["git", "clone", remote, str(repo_dir)]
    elif (repo_dir / ".git").is_dir():
        command = ["git", "-C", str(repo_dir), "pull", "--ff-only", remote]
    else:
        raise RuntimeError(f"{repo_dir} exists but is not a Git checkout.")
    result = subprocess.run(command, env=git_env, capture_output=True, text=True)
    if result.returncode:
        detail = (result.stderr + result.stdout).replace(token, "[REDACTED]").replace(auth, "[REDACTED]")
        raise RuntimeError(
            "Repository setup failed. Check Kaggle Internet access, GITHUB_TOKEN "
            "secret access for this notebook, and repository permissions.\n" + detail
        )
finally:
    del token, auth, git_env

src_dir = repo_dir / "src"
required = ("model.py", "prw_dataset.py", "training.py", "prw_evaluation.py", "dino_reid.py", "dinov3_reid.py")
missing = [name for name in required if not (src_dir / name).is_file()]
if missing:
    raise RuntimeError(f"Repository at {repo_dir} is missing source files: {missing}")
src_path = str(src_dir.resolve())
if src_path in sys.path:
    sys.path.remove(src_path)
sys.path.insert(0, src_path)
import importlib
importlib.invalidate_caches()
print(f"Project imports configured from {src_path}")


In [ ]:


from kaggle_secrets import UserSecretsClient
import wandb

secrets = UserSecretsClient()
wandb_api_key = secrets.get_secret("WAND_B_API_KEY")

wandb.login(key=wandb_api_key)




PRW_ROOT = "/kaggle/input/datasets/edoardomerli/prw-person-re-identification-in-the-wild"

data_root = PRW_ROOT



In [ ]:
import random
import torch
import wandb
from kaggle_secrets import UserSecretsClient
from torch.utils.data import DataLoader

from model import PersonSearchModel
from prw_dataset import (PRWPersonSearchDataset, PRWSubset,
                         PKIdentityBatchSampler, prw_collate_fn)
from training import make_grad_scaler, start_run, train_one_epoch, validate


SEED = 0
torch.manual_seed(SEED)
random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))
    print(f'GPU memory: {torch.cuda.get_device_properties(0).total_memory / 2**30:.2f} GiB')

In [ ]:
from dino_reid import DinoPersonSearch
from dinov3_reid import DinoV3CropReID, FrozenDinoV3CLS

DINO_CHECKPOINT = Path('/kaggle/input/YOUR-DINO-EXPERIMENT/best.pt')
checkpoint = torch.load(DINO_CHECKPOINT, map_location='cpu', weights_only=False)
if checkpoint['run_config']['experiment'] != 'swin_frozen_dinov3_frozen_crop_triplet':
    raise ValueError('Expected the frozen-DINOv3 experiment checkpoint')
config = dict(checkpoint['detector_config'])
config['pretrained'] = False
detector = PersonSearchModel(**config)
detector.load_state_dict(checkpoint['detector_state_dict'], strict=True)
projected_reid = DinoV3CropReID(**checkpoint['reid_config'], pretrained=False)
projected_reid.load_state_dict(checkpoint['reid_state_dict'], strict=True)
raw_reid = FrozenDinoV3CLS(projected_reid.backbone)
model = DinoPersonSearch(detector, raw_reid).to(device).eval()
provenance = {
    'test/checkpoint_path': str(DINO_CHECKPOINT),
    'test/checkpoint_epoch': checkpoint['epoch'],
    'test/backbone': 'frozen dinov3_vits16 + frozen swin_t detector',
    'test/metric_loss': 'none: raw CLS evaluation',
    'test/embedding_mode': 'raw_cls_no_projection',
    'test/embedding_dim': raw_reid.embedding_proj.out_features,
    'test/detector_epoch': checkpoint['run_config']['detector_epoch'],
    'test/score_threshold': .5,
}
del checkpoint, projected_reid
if wandb.run is not None:
    wandb.finish()
run = wandb.init(entity='models-alma-mater-studio', project='person-search',
                 name='prw-frozen-dinov3-raw-cls-evaluation', job_type='evaluation',
                 config=provenance)
print('Evaluating:', provenance)


In [ ]:
import importlib
import prw_evaluation
importlib.reload(prw_evaluation)
from prw_evaluation import evaluate_prw, make_wandb_retrieval_table

run = wandb.run
# An explicit evaluation axis gives test metrics a chart point on resumed runs too.
evaluation_index = int(run.summary.get("test/evaluation_index", 0)) + 1
run.define_metric("test/evaluation_index")
run.define_metric("test/*", step_metric="test/evaluation_index")
retrieval_metrics = evaluate_prw(
    model, PRW_ROOT, device, batch_size=2, score_threshold=0.5,
    iou_threshold=0.5, return_records=True, table_queries=100, table_top_k=5,
    query_batch_size=64, ranking_batch_size=8, gallery_crop_batch_size=32,
)
table_records = retrieval_metrics.pop("retrieval_records")
test_summary = {
    "test/mAP": retrieval_metrics["retrieval_mAP"],
    "test/top1": retrieval_metrics["retrieval_top1"],
    "test/evaluated_queries": retrieval_metrics["retrieval_queries"],
    "test/gallery_frames": retrieval_metrics["retrieval_gallery_frames"],
    "test/detections": retrieval_metrics["retrieval_detections"],
    "test/evaluation_index": evaluation_index,
}
# Persist scalar results BEFORE creating/uploading optional media.
import json
results_dir = Path("/kaggle/working/evaluation_results")
results_dir.mkdir(parents=True, exist_ok=True)
results_path = results_dir / f"{run.id}-evaluation-{evaluation_index}.json"
results_path.write_text(json.dumps({**test_summary, **provenance}, indent=2))
for metric in ("test/mAP", "test/top1"):
    run.define_metric(metric, step_metric="test/evaluation_index", summary="copy")
for key, value in {**test_summary, **provenance}.items():
    run.summary[key] = value
run.log({**test_summary, **provenance}, commit=True)
# Explicit table and bar chart: a single test result is easy to miss on line plots.
final_scores = wandb.Table(
    columns=["metric", "value", "percent", "checkpoint_epoch", "backbone", "loss"],
    data=[[name, test_summary[key], 100 * test_summary[key],
           provenance["test/checkpoint_epoch"], provenance["test/backbone"], provenance["test/metric_loss"]]
          for name, key in (("mAP", "test/mAP"), ("Top-1", "test/top1"))],
)
run.log({
    "test/evaluation_index": evaluation_index,
    "test/final_scores": final_scores,
    "test/final_scores_chart": wandb.plot.bar(
        final_scores, "metric", "percent", title="Final PRW test scores (%)"),
})
from IPython.display import display
display(final_scores.get_dataframe())
print(f"FINAL PRW TEST: mAP={test_summary['test/mAP']:.4f}, top-1={test_summary['test/top1']:.4f}")
print(f"Scalar results logged to {run.url}; local backup: {results_path}")

try:
    retrieval_table = make_wandb_retrieval_table(table_records, PRW_ROOT)
    # Log explicit image comparisons as well as the interactive table.
    # Convert rendered figures to raster images so W&B displays an image gallery.
    import io
    import matplotlib.pyplot as plt
    from PIL import Image
    from prw_evaluation import plot_retrieval_query
    
    PREVIEW_QUERIES = 10  # table still includes up to 100 queries
    comparison_images = []
    for query_index in range(min(PREVIEW_QUERIES, len(table_records))):
        fig = plot_retrieval_query(table_records, PRW_ROOT, query_index=query_index, top_k=5)
        try:
            with io.BytesIO() as buffer:
                fig.savefig(buffer, format="png", dpi=100, bbox_inches="tight")
                buffer.seek(0)
                with Image.open(buffer) as rendered:
                    comparison_images.append(wandb.Image(
                        rendered.convert("RGB"),
                        caption=f"Query {query_index}: ID {table_records[query_index]['identity']}",
                    ))
            if query_index == 0:
                plt.show()  # display one comparison in this evaluation cell too
        finally:
            plt.close(fig)
    
    payload = {"test/evaluation_index": evaluation_index, "test/query_top5_retrievals": retrieval_table}
    if comparison_images:
        payload["test/query_top5_images"] = comparison_images
    else:
        print("No retrieval records were returned; no comparison images to log.")
    run.log(payload)
    print(f"Logged {len(table_records)} query rows and {len(comparison_images)} comparison images to {run.url}")
finally:
    # Finish/sync the scalar record even if table rendering or media logging fails.
    run.finish()


## Inspect query and top-five predictions

The W&B table above contains one row per query. This cell also displays the freshly evaluated results inline. Change `QUERY_INDEX` to browse; rerunning this plot does not repeat evaluation or require an active W&B run.


In [ ]:
import matplotlib.pyplot as plt
from prw_evaluation import plot_retrieval_query

QUERY_INDEX = 0
TOP_K = 5
if table_records:
    fig = plot_retrieval_query(table_records, PRW_ROOT, query_index=QUERY_INDEX, top_k=TOP_K)
    plt.show()
else:
    print("No retrieval records available for visualization.")
